# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [5]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        self.start = (0, 0)

        self.walls = {
            (0, 3),
            (1, 1),
            (2, 4),
            (4, 2),
        }

        self.slippery_states = {
            (1, 2),
            (2, 1),
            (3, 3),
        }

        self.terminal_states = {
            (0, 5): 10.0,   # entrega
            (2, 2): 2.0,    # carga
            (3, 5): -10.0,  # peligro mortal
        }

        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        r, c = state
        if not (0 <= r < self.height and 0 <= c < self.width):
            return False
        if state in self.walls:
            return False
        return True

    def states(self):
        return [
            (r, c)
            for r in range(self.height)
            for c in range(self.width)
            if (r, c) not in self.walls
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        Recuerda:
        - las probabilidades dependen de si 'state' es resbaloso;
        - si golpea pared/borde, next_state = state.
        """
        # Estados terminales no tienen dinámica: quedarse ahí con prob 1
        if self.is_terminal(state):
            return [(state, 1.0)]

        # Probabilidades según tipo de piso
        if state in self.slippery_states:
            p_intended, p_dev = 0.60, 0.20
        else:
            p_intended, p_dev = 0.90, 0.05

        # Direcciones de desviación (perpendiculares a la acción)
        # UP/DOWN -> desviaciones LEFT/RIGHT ; LEFT/RIGHT -> desviaciones UP/DOWN
        dr, dc = action
        if dr != 0:  # movimiento vertical (UP o DOWN)
            left_dev  = (0, -1)
            right_dev = (0, 1)
        else:        # movimiento horizontal (LEFT o RIGHT)
            left_dev  = (-1, 0)
            right_dev = (1, 0)

        outcomes = [
            (action, p_intended),
            (left_dev, p_dev),
            (right_dev, p_dev),
        ]

        # Acumular probabilidades por next_state (por si dos resultan en el mismo lugar)
        result = {}
        for (dr_, dc_), p in outcomes:
            r, c = state
            next_state = (r + dr_, c + dc_)
            if not self.is_valid_state(next_state):
                next_state = state  # choca con pared o borde -> se queda
            result[next_state] = result.get(next_state, 0.0) + p

        return list(result.items())


### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [6]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [11]:
def expected_next_value(grid, state, action, V):
    # sum_{s'} T(s,a,s') V(s')
    transitions = grid.get_transition_probs(state, action)
    return sum(p * V[next_state] for next_state, p in transitions)


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    S = grid.states()
    V = {s: 0.0 for s in S}

    for i in range(max_iter):
        delta = 0.0
        V_new = {}

        for s in S:
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
            else:
                q_values = [
                    expected_next_value(grid, s, a, V)
                    for a in grid.actions
                ]
                V_new[s] = grid.get_reward(s) + grid.gamma * max(q_values)

            delta = max(delta, abs(V_new[s] - V[s]))

        V = V_new

        if delta < threshold:
            return V, i + 1

    return V, max_iter


def extract_policy(grid, V):
    # pi*(s) = argmax_a sum T(s,a,s') V(s')
    policy = {}

    for s in grid.states():
        if grid.is_terminal(s):
            continue

        best_action = None
        best_value = float("-inf")

        for a in grid.actions:
            q = expected_next_value(grid, s, a, V)
            if q > best_value:
                best_value = q
                best_action = a

        policy[s] = best_action

    return policy


## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [12]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    S = grid.states()
    V = {s: 0.0 for s in S}

    for i in range(max_iter):
        delta = 0.0
        V_new = {}

        for s in S:
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
            else:
                a = policy[s]
                V_new[s] = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, a, V)

            delta = max(delta, abs(V_new[s] - V[s]))

        V = V_new

        if delta < threshold:
            break

    return V


def policy_improvement(grid, V):
    # pi_new(s) = argmax_a sum T(s,a,s') V(s')
    policy = {}

    for s in grid.states():
        if grid.is_terminal(s):
            continue

        best_action = None
        best_value = float("-inf")

        for a in grid.actions:
            q = expected_next_value(grid, s, a, V)
            if q > best_value:
                best_value = q
                best_action = a

        policy[s] = best_action

    return policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    S = grid.states()
    non_terminal_states = [s for s in S if not grid.is_terminal(s)]

    # 1. Política inicial arbitraria (siempre la primera acción disponible)
    policy = {s: grid.actions[0] for s in non_terminal_states}

    history = []

    for i in range(max_iter):
        # 2. Evaluación
        V = policy_evaluation(grid, policy, threshold=threshold)

        # 3. Mejora
        new_policy = policy_improvement(grid, V)

        # ¿Cuántos estados cambiaron de acción? (para la historia/diagnóstico)
        n_changes = sum(
            1 for s in non_terminal_states if new_policy[s] != policy[s]
        )
        history.append(n_changes)

        # 4. ¿Estabilidad?
        if n_changes == 0:
            return new_policy, V, history

        policy = new_policy

    return policy, V, history


## Parte 4 — Visualización y comparación


In [13]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [14]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [16, 5, 1, 0]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**? Prefiere +2, ignora por completo +10.
2. ¿Por qué una recompensa menor podría ser óptima? Porque el costo por paso -1 y el riesgo de pasar cerca de peligros hacen que la recompensa lejana +10 "cueste" más de lo que vale.
3. ¿En qué estados el piso resbaloso cambia la decisión? En las celdas resbalosas vecinas a peligros, como (2,1) y (3,3)
4. ¿Qué papel cumple el costo por paso `-1`? Actúa como "impuesto por tiempo": cada paso extra resta valor, así que empuja al robot a preferir caminos cortos aunque la recompensa final sea menor.
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados? Porque ahora hay dos tipos de piso (normal y resbaloso) con probabilidades distintas, entonces T(s,a,s') depende del estado s, no puede ser una tabla única fija para todo el grid.

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


**Experimento A**

In [15]:
# Experimento A — Menos costo por paso
grid_exp_a = WarehouseMDP()
grid_exp_a.living_reward = -0.1

V_vi_a, n_vi_a = value_iteration(grid_exp_a)
pi_vi_a = extract_policy(grid_exp_a, V_vi_a)

print("=== EXPERIMENTO A: living_reward = -0.1 ===")
print("Iteraciones:", n_vi_a)
print("\nValores:")
print_values(grid_exp_a, V_vi_a)
print("\nPolítica:")
print_policy(grid_exp_a, pi_vi_a)

=== EXPERIMENTO A: living_reward = -0.1 ===
Iteraciones: 26

Valores:
 +1.649 |  +1.992 |  +2.361 |   WALL   |  +8.591 | +10.000
 +1.358 |   WALL   |  +2.797 |  +3.911 |  +4.550 |  +8.591
 +1.259 |  +1.533 |  +2.000 |  +3.306 |   WALL   |  +7.537
 +1.243 |  +1.540 |  +2.040 |  +2.417 |  +2.026 | -10.000
 +0.865 |  -1.794 |   WALL   |  +2.026 |  +1.709 |  +0.874

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↑  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  →  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 


**Experimento B**

In [21]:
class WarehouseMDP:
    def __init__(self, p_slippery_intended=0.60, p_slippery_dev=0.20):
        self.height = 5
        self.width = 6

        self.start = (0, 0)

        self.walls = {
            (0, 3),
            (1, 1),
            (2, 4),
            (4, 2),
        }

        self.slippery_states = {
            (1, 2),
            (2, 1),
            (3, 3),
        }

        self.terminal_states = {
            (0, 5): 10.0,
            (2, 2): 2.0,
            (3, 5): -10.0,
        }

        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.p_slippery_intended = p_slippery_intended
        self.p_slippery_dev = p_slippery_dev

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        r, c = state
        if not (0 <= r < self.height and 0 <= c < self.width):
            return False
        if state in self.walls:
            return False
        return True

    def states(self):
        return [
            (r, c)
            for r in range(self.height)
            for c in range(self.width)
            if (r, c) not in self.walls
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        if self.is_terminal(state):
            return [(state, 1.0)]

        if state in self.slippery_states:
            p_intended, p_dev = self.p_slippery_intended, self.p_slippery_dev
        else:
            p_intended, p_dev = 0.90, 0.05

        dr, dc = action
        if dr != 0:
            left_dev  = (0, -1)
            right_dev = (0, 1)
        else:
            left_dev  = (-1, 0)
            right_dev = (1, 0)

        outcomes = [
            (action, p_intended),
            (left_dev, p_dev),
            (right_dev, p_dev),
        ]

        result = {}
        for (dr_, dc_), p in outcomes:
            r, c = state
            next_state = (r + dr_, c + dc_)
            if not self.is_valid_state(next_state):
                next_state = state
            result[next_state] = result.get(next_state, 0.0) + p

        return list(result.items())

In [22]:
# Experimento B — Piso muy resbaloso
grid_exp_b = WarehouseMDP(p_slippery_intended=0.40, p_slippery_dev=0.30)

# Validación rápida de que las probabilidades siguen sumando 1
for s in grid_exp_b.states():
    for a in grid_exp_b.actions:
        total = sum(p for _, p in grid_exp_b.get_transition_probs(s, a))
        assert abs(total - 1.0) < 1e-12
print("✓ Transiciones válidas para Experimento B.")

V_vi_b, n_vi_b = value_iteration(grid_exp_b)
pi_vi_b = extract_policy(grid_exp_b, V_vi_b)

print("\n=== EXPERIMENTO B: piso resbaloso 0.40/0.30/0.30 ===")
print("Iteraciones:", n_vi_b)
print("\nValores:")
print_values(grid_exp_b, V_vi_b)
print("\nPolítica:")
print_policy(grid_exp_b, pi_vi_b)

✓ Transiciones válidas para Experimento B.

=== EXPERIMENTO B: piso resbaloso 0.40/0.30/0.30 ===
Iteraciones: 22

Valores:
 -2.706 |  -1.809 |  -0.797 |   WALL   |  +7.607 | +10.000
 -2.652 |   WALL   |  +0.395 |  +2.104 |  +3.670 |  +7.607
 -1.744 |  -0.670 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.831 |  -0.774 |  +0.534 |  -1.137 |  -2.152 | -10.000
 -2.785 |  -3.929 |   WALL   |  -2.152 |  -2.974 |  -4.041

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 


**Experimento C**

In [23]:
# Experimento C — Más paciencia
grid_exp_c = WarehouseMDP()
grid_exp_c.gamma = 0.99

V_vi_c, n_vi_c = value_iteration(grid_exp_c)
pi_vi_c = extract_policy(grid_exp_c, V_vi_c)

print("=== EXPERIMENTO C: gamma = 0.99 ===")
print("Iteraciones:", n_vi_c)
print("\nValores:")
print_values(grid_exp_c, V_vi_c)
print("\nPolítica:")
print_policy(grid_exp_c, pi_vi_c)

=== EXPERIMENTO C: gamma = 0.99 ===
Iteraciones: 24

Valores:
 -1.456 |  -0.310 |  +0.809 |   WALL   |  +8.601 | +10.000
 -2.179 |   WALL   |  +2.003 |  +4.118 |  +5.354 |  +8.601
 -1.081 |  +0.119 |  +2.000 |  +2.913 |   WALL   |  +7.395
 -1.606 |  -0.467 |  +0.799 |  +0.817 |  -0.359 | -10.000
 -2.752 |  -3.737 |   WALL   |  -0.359 |  -1.408 |  -2.892

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 


Sí — con `gamma=0.99` el robot se vuelve más paciente y ahora prefiere recorrer más pasos para llegar a la entrega **+10** (por ejemplo, `(1,2)` cambia de ir directo a `+2` a rodear hacia `+10`), en vez de conformarse con la carga +2 cercana como en el baseline.

**Bonus**

In [24]:
# Bonus — encontrar el punto de cambio de living_reward

def start_prefers_delivery(grid, living_reward):
    """
    Reconstruye el grid con un living_reward dado y revisa si,
    siguiendo la política óptima desde START, el robot termina
    llegando a la entrega +10 en vez de a la carga +2.
    """
    grid.living_reward = living_reward
    V, _ = value_iteration(grid)
    policy = extract_policy(grid, V)

    # Simulamos el camino determinista siguiendo pi*(s) desde START
    state = grid.start
    visited = set()
    while not grid.is_terminal(state):
        if state in visited:  # por seguridad, evita loops infinitos
            return None
        visited.add(state)
        action = policy[state]
        # tomamos el "next_state" más probable (la acción intendida)
        r, c = state
        dr, dc = action
        next_state = (r + dr, c + dc)
        state = next_state if grid.is_valid_state(next_state) else state

    reward_final = grid.terminal_states[state]
    return reward_final == 10.0  # True si llegó a +10, False si llegó a +2/-10


# Barrido grueso para ubicar el rango donde cambia la política
grid_bonus = WarehouseMDP()

print("Barrido de living_reward:")
print(f"{'living_reward':>15} | {'llega a +10?':>12}")
for lr in np.arange(-1.0, -0.0, 0.05):
    prefers_10 = start_prefers_delivery(grid_bonus, lr)
    print(f"{lr:15.2f} | {str(prefers_10):>12}")

Barrido de living_reward:
  living_reward | llega a +10?
          -1.00 |        False
          -0.95 |        False
          -0.90 |        False
          -0.85 |        False
          -0.80 |        False
          -0.75 |         True
          -0.70 |         True
          -0.65 |         True
          -0.60 |         True
          -0.55 |         True
          -0.50 |         True
          -0.45 |         True
          -0.40 |         True
          -0.35 |         True
          -0.30 |         True
          -0.25 |         True
          -0.20 |         True
          -0.15 |         True
          -0.10 |         True
          -0.05 |         True


In [25]:
# Refinamiento con búsqueda binaria una vez identificado el rango aproximado
lo, hi = -0.50, -0.30  # AJUSTA estos límites según lo que veas en el barrido de arriba

while hi - lo > 1e-3:
    mid = (lo + hi) / 2
    if start_prefers_delivery(grid_bonus, mid):
        hi = mid  # con mid ya prefiere +10 -> el punto de cambio está más cerca de mid o antes
    else:
        lo = mid  # con mid todavía prefiere +2 -> el punto de cambio está después de mid

print(f"\nEl cambio de política ocurre aproximadamente en living_reward ≈ {(lo+hi)/2:.4f}")


El cambio de política ocurre aproximadamente en living_reward ≈ -0.4996
